# EDA Kesiapan Data & Pemilihan Fitur - Prediksi Kerusakan PART 30 Hari (OMEXP)

Notebook ini memeriksa apakah data cukup bersih dan aman dipakai untuk melatih
model prediksi kerusakan 30 hari: kualitas pencatatan, kebocoran data masa
depan, ketidakseimbangan target, korelasi/redundansi fitur, drift, dan
keputusan akhir fitur mana yang dipakai.

Untuk pertanyaan operasional (barang paling sering rusak, tren, efektivitas
perbaikan, dan lain-lain), lihat notebook terpisah `01a_business_eda.ipynb`.


In [ ]:
from pathlib import Path
import os, sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import Markdown, display
PROJECT_DIR = Path.cwd() if (Path.cwd() / 'src').exists() else Path.cwd().parent
sys.path.insert(0, str(PROJECT_DIR / 'src'))
from database import connect
sns.set_theme(style='whitegrid')
def query(sql, params=None):
    with connect() as conn:
        with conn.cursor() as cur:
            cur.execute(sql, params or ())
            return pd.DataFrame(cur.fetchall(), columns=[d.name for d in cur.description])

## 0. Tujuan dan pertanyaan yang dijawab

**Tujuan:** menilai kesiapan data untuk modeling dan menentukan fitur yang
aman serta layak dipakai pada baseline model prediksi kerusakan 30 hari.

Pertanyaan yang dijawab:

1. Apakah jumlah dan kualitas datanya cukup?
2. Apa definisi target dan seberapa seimbang labelnya?
3. Fitur apa yang tersedia, redundan, atau berisiko memakai data masa depan?
4. Apakah pola dan fitur stabil dari waktu ke waktu?
5. Data atau fitur apa yang belum boleh dipakai?

Setiap bagian mengikuti pola: **pertanyaan -> alasan -> metrik -> hasil ->
interpretasi -> keputusan**.


## 1. Konteks bisnis dan definisi data

Konteks utama adalah PART yang berpindah melalui event operasional. TERMINAL
dipisahkan karena populasi failure-nya berbeda. Definisi berikut digunakan konsisten
di SQL, EDA, dan dataset snapshot.

| Istilah | Arti sederhana |
|---|---|
| **Model item** | Jenis atau tipe barang. Satu model dapat dimiliki banyak unit fisik. |
| **Unit item** | Satu barang fisik yang dibedakan dengan identifier atau serial. |
| **PART** | Komponen yang dipasang pada perangkat atau lokasi dan menjadi objek utama prediksi failure. |
| **TERMINAL** | Perangkat terminal. Jumlah dan pola failure-nya dipisahkan dari PART. |
| **Master data** | Daftar resmi model, lokasi, klien, status, dan jenis pekerjaan yang menjadi acuan validasi. |
| **Journey/event** | Satu catatan kegiatan atau perubahan status sebuah item pada suatu waktu. |
| **Operational event** | Event dengan waktu yang layak dipakai. RECON administratif dan tanggal tidak valid tidak ikut menghitung durasi. |
| **Installation cycle** | Periode sejak PART dipasang sampai failure pertama, pemasangan berikutnya, atau akhir data. |
| **Failure onset** | Waktu awal PART dianggap berhenti beroperasi karena kerusakan. |
| **Failure outcome** | Status yang mengonfirmasi hasil kerusakan, misalnya BROKEN atau UNREPAIRABLE. |
| **Snapshot** | Foto kondisi data PART pada satu tanggal, bukan foto gambar. Satu PART dapat memiliki banyak snapshot. |
| **Target 30 hari** | Pertanyaan apakah failure terjadi setelah snapshot dan paling lambat 30 hari berikutnya. |
| **Positif** | Snapshot yang benar-benar diikuti failure dalam 30 hari. |
| **Negatif** | Snapshot yang tidak diikuti failure dan memiliki bukti follow-up penuh selama 30 hari. |
| **Positive rate** | Jumlah positif dibagi seluruh observasi yang layak. Nilainya berubah menurut unit hitung: snapshot, cycle, atau unit PART. |
| **Class imbalance** | Jumlah label positif dan negatif sangat tidak seimbang. Akurasi biasa dapat terlihat bagus walaupun model gagal menemukan failure. |
| **Korelasi** | Ukuran apakah dua fitur cenderung berubah bersama. Korelasi tinggi dapat menunjukkan informasi yang berulang. |
| **Multikolinearitas/redundansi** | Beberapa fitur membawa informasi yang hampir sama, misalnya hitungan 90 hari dan 180 hari. Tidak selalu salah, tetapi perlu diseleksi atau diregularisasi. |
| **Information Value (IV)** | Screening satu fitur terhadap target. Nilai lebih besar berarti pemisahan awal lebih kuat, tetapi bukan bukti kausal atau feature importance model final. |
| **Drift / PSI** | Perubahan distribusi fitur terhadap periode referensi. PSI di bawah 0,10 relatif stabil; 0,10-0,25 perlu dipantau; minimal 0,25 menunjukkan drift besar. |
| **Missing struktural** | Nilai kosong yang mempunyai arti, misalnya hari sejak failure terakhir kosong karena PART memang belum pernah failure. |
| **Follow-up tidak lengkap** | Belum tersedia cukup data setelah snapshot atau failure untuk memastikan kejadian selanjutnya. |
| **Right-censored** | Cycle masih berjalan ketika database berhenti; akhir umur PART belum diketahui. |
| **Cohort valid** | Kelompok cycle PART yang lolos aturan awal: identifier, model, dan waktu pemasangan dapat dipercaya. |
| **Fitur** | Informasi yang tersedia pada tanggal snapshot dan dapat menjadi masukan model, misalnya umur PART, model, lokasi, atau corrective sebelumnya. |
| **Lokasi canonical** | Nama lokasi yang sudah dicocokkan dengan master lokasi resmi melalui exact match, alias kontekstual terverifikasi, atau fuzzy berkeyakinan tinggi. |
| **Fuzzy score** | Nilai kemiripan teks 0-100%. Auto-mapping memerlukan skor minimal 90% dan selisih minimal 8 poin dari kandidat kedua. |
| **Fuzzy review** | Kandidat yang belum aman dipetakan otomatis karena skor rendah atau dua kandidat terlalu berdekatan. |
| **Data leakage** | Kesalahan ketika informasi masa depan ikut dipakai untuk memprediksi masa depan. |


## 2. Data preparation dan normalisasi

**Pertanyaan:** bagaimana event mentah menjadi dataset observasi yang dapat
diaudit? **Mengapa penting:** kesalahan mapping atau semantic dapat mengubah
urutan waktu dan ground truth.

```text
Data mentah -> cleaning teks -> canonical/fuzzy mapping -> validasi master
            -> semantic event -> operational timeline -> installation cycle
            -> observation dataset 30 hari
```

| Kondisi | Perlakuan pipeline | Keputusan |
|---|---|---|
| RECON | Disimpan untuk audit, tidak ikut durasi operasional | KEEP_AUDIT |
| Tanggal invalid/future | Tidak masuk operational timeline | EXCLUDE_TIME |
| Lokasi fuzzy tidak aman | Nilai mentah disimpan, tidak dipakai sebagai fitur lokasi | REVIEW |
| Model inconsistent | Tidak masuk initial cohort | EXCLUDE_COHORT |
| Failure incomplete flow | Tetap failure, bukan negative | KEEP_POSITIVE_REVIEW |
| Reinstall tanpa failure tercatat | Ditandai unknown, tidak otomatis menjadi negatif | EXCLUDE_NEGATIVE |
| Right-censored dengan coverage aktivitas tidak terkonfirmasi | Tetap tersedia untuk audit dan sensitivity analysis | REVIEW_COVERAGE |

Alias yang disetujui dan singkatan teks disimpan pada tabel mapping `analytics`,
bukan hard-coded di fungsi. Nilai mentah, canonical, metode, dasar mapping, dan
approval tetap dapat ditelusuri.


## 3. Data quality, coverage, dan keputusan cleaning

**Pertanyaan:** masalah apa yang ditemukan, seberapa besar dampaknya, dan apa
tindakan pipeline? Bagian ini menggabungkan readiness, audit journal, master
coverage, fuzzy review, outlier, serta kualitas cycle/label negatif sebelum
analisis pola dilakukan.


### 3.1 Readiness snapshot dan pemeriksaan leakage

**Metrik:** cycle valid, snapshot eligible, key ganda, label positif invalid, dan
fitur waktu negatif. Nilai nol pada tiga pemeriksaan terakhir berarti struktur
snapshot lolos quality gate dasar.


In [ ]:
readiness = query('SELECT * FROM analytics.eda_failure_readiness_summary ORDER BY metric')
metric_labels = {'all_observations': 'Semua snapshot', 'cycles_with_failure': 'Siklus yang mengalami kerusakan', 'excluded_incomplete_followup': 'Snapshot tanpa data 30 hari yang lengkap', 'installation_cycles': 'Semua siklus pemasangan', 'invalid_zero_duration_cycles': 'Siklus berdurasi nol (tidak valid)', 'positive_30d_observations': 'Snapshot diikuti kerusakan dalam 30 hari', 'right_censored_cycles': 'Siklus masih berjalan saat data berakhir', 'training_eligible_observations': 'Snapshot yang dapat dipakai untuk model', 'valid_model_cohort_cycles': 'Siklus PART yang lolos pemeriksaan awal'}
readiness_display = readiness.assign(metric=readiness['metric'].replace(metric_labels)).rename(columns={'metric': 'Ukuran yang diperiksa', 'value': 'Jumlah'})
display(readiness_display)
checks = query("""
SELECT
 COUNT(*) - COUNT(DISTINCT (installation_cycle_id, observation_on)) AS duplicate_keys,
 COUNT(*) FILTER (WHERE target_failure_30d AND NOT (next_failure_on > observation_on AND next_failure_on <= observation_on + INTERVAL '30 days')) AS invalid_positive_labels,
 COUNT(*) FILTER (WHERE days_since_installation < 0 OR days_since_last_event < 0 OR days_since_last_failure < 0) AS negative_time_features
FROM analytics.item_observation_30d
""")
display(checks.rename(columns={'duplicate_keys': 'Snapshot ganda', 'invalid_positive_labels': 'Label kerusakan tidak sesuai', 'negative_time_features': 'Perhitungan waktu negatif'}))
assert checks.iloc[0].eq(0).all(), 'Dataset gagal pemeriksaan leakage/key.'
eligible_metric = int(readiness.loc[readiness.metric.eq('training_eligible_observations'),'value'].iloc[0])
display(Markdown(f"""**Temuan utama:** tersedia **{eligible_metric:,} snapshot** yang lolos aturan label saat ini dan seluruh pemeriksaan key/leakage bernilai nol.

**Tindak lanjut:** snapshot tetap harus mengikuti split waktu dan feature whitelist; lolos quality gate ini belum berarti siap produksi.""".replace(',', '.')))


### 3.2 Audit journal, duplikasi, dan mapping fuzzy

**Metrik:** jumlah record terdampak, persentase terhadap journal, dampak, serta
keputusan keep/exclude/review. Detail fuzzy menunjukkan nilai sumber, kandidat,
skor, margin, dan metode keputusan.


In [ ]:
journal_quality = query('SELECT * FROM analytics.eda_journey_quality_summary ORDER BY check_order')
quality_labels = {'MISSING_ITEM_IDENTIFIER': 'Identifier item kosong', 'MISSING_ITEM_MODEL': 'Model item kosong', 'MISSING_ITEM_TYPE': 'Tipe item kosong', 'MISSING_ITEM_CATEGORY': 'Kategori item kosong', 'MISSING_CLIENT': 'Klien kosong', 'MISSING_LOCATION': 'Lokasi mentah kosong', 'LOCATION_NOT_IN_MASTER': 'Lokasi belum cocok setelah mapping', 'CLIENT_NOT_IN_MASTER': 'Klien belum cocok setelah mapping', 'MISSING_STATUS': 'Status kosong', 'MISSING_DATE': 'Tanggal kosong', 'INVALID_OR_FUTURE_DATE': 'Tanggal invalid/masa depan', 'DUPLICATE_JOURNEY_ID': 'journey_id duplikat', 'EXACT_LOG_DUPLICATE_EXTRA_ROWS': 'Baris tambahan dengan isi log identik', 'CREATED_ON_NOT_DATETIME': 'created_on bukan datetime'}
journal_quality['affected_count'] = pd.to_numeric(journal_quality['affected_count'], errors='coerce').fillna(0).astype(int)
journal_quality_display = journal_quality.assign(check_name=journal_quality['check_name'].replace(quality_labels)).rename(columns={'check_name': 'Pemeriksaan', 'affected_count': 'Jumlah terdampak', 'explanation': 'Arti'})
display(journal_quality_display[['Pemeriksaan', 'Jumlah terdampak', 'Arti']])
quality_nonzero = journal_quality_display[journal_quality_display['Jumlah terdampak'].gt(0)]
if not quality_nonzero.empty:
    plt.figure(figsize=(9, 4)); sns.barplot(data=quality_nonzero, y='Pemeriksaan', x='Jumlah terdampak', color='indianred'); plt.xscale('log'); plt.title('Masalah kualitas journal yang benar-benar ditemukan'); plt.xlabel('Jumlah terdampak (skala log)'); plt.ylabel(''); plt.tight_layout(); plt.show()
date_types = query("""SELECT table_schema, table_name, column_name, data_type FROM information_schema.columns WHERE (table_schema, table_name, column_name) IN (('journal','t_item_journey','created_on'), ('analytics','item_journey_clean','created_on')) ORDER BY table_schema""")
display(date_types.rename(columns={'table_schema': 'Layer', 'table_name': 'Tabel/view', 'column_name': 'Kolom', 'data_type': 'Tipe data'}))
duplicate_groups = query("""SELECT ARRAY_AGG(journey_id ORDER BY journey_id)::text journey_ids, item_model_code_clean, status_clean, activity_clean, created_on, place_clean, COUNT(*) row_count FROM analytics.item_journey_clean GROUP BY item_identifier_clean, created_on, item_category_clean, item_type_clean, item_model_code_clean, client_clean, ref_doc_code_clean, wo_type_clean, wo_code_clean, place_clean, activity_clean, status_clean, done_by_clean, remark HAVING COUNT(*) > 1 ORDER BY row_count DESC, created_on LIMIT 20""")
display(duplicate_groups.rename(columns={'journey_ids': 'journey_id yang perlu dibandingkan', 'item_model_code_clean': 'Model', 'status_clean': 'Status', 'activity_clean': 'Aktivitas', 'created_on': 'Waktu', 'place_clean': 'Lokasi mentah', 'row_count': 'Jumlah baris'}))
fuzzy_mapping = query('SELECT * FROM analytics.eda_fuzzy_mapping_review ORDER BY mapping_type, event_count DESC')
fuzzy_mapping['similarity_percentage'] = (pd.to_numeric(fuzzy_mapping.similarity_score, errors='coerce') * 100).round(2); fuzzy_mapping['margin_percentage'] = (pd.to_numeric(fuzzy_mapping.score_margin, errors='coerce') * 100).round(2)
display(fuzzy_mapping.rename(columns={'mapping_type': 'Jenis', 'source_value': 'Nilai sumber', 'canonical_value': 'Nilai canonical', 'best_candidate': 'Kandidat terbaik', 'mapping_method': 'Keputusan', 'similarity_percentage': 'Kemiripan (%)', 'margin_percentage': 'Selisih kandidat (%)', 'event_count': 'Event', 'item_count': 'Item'})[['Jenis','Nilai sumber','Nilai canonical','Kandidat terbaik','Keputusan','Kemiripan (%)','Selisih kandidat (%)','Event','Item']])
display(Markdown(f"""**Temuan utama:** ada **{len(quality_nonzero)} jenis pemeriksaan** dengan dampak non-zero. Exact duplicate tidak dihapus otomatis dan mapping ambigu tetap masuk review.

**Tindak lanjut:** hanya record dengan aturan exclusion eksplisit yang dikeluarkan; record review tetap disimpan untuk audit."""))


### 3.3 Coverage master dan missing data

**Metrik:** unmatched snapshot, kategori langka, dan missing rate. Missing
struktural seperti â€œbelum pernah failureâ€ tidak dianggap kesalahan input dan
tidak boleh diisi nol tanpa indikator.


In [ ]:
master_coverage = query('SELECT * FROM analytics.eda_snapshot_master_coverage ORDER BY feature_name')
for col in ['total_snapshot','matched_snapshot','unmatched_snapshot','category_count','rare_category_count','rare_snapshot_count','unmatched_percentage','rare_snapshot_percentage']: master_coverage[col] = pd.to_numeric(master_coverage[col], errors='coerce')
coverage_display = master_coverage.rename(columns={'feature_name': 'Fitur master', 'total_snapshot': 'Snapshot training', 'matched_snapshot': 'Berhasil dipetakan', 'unmatched_snapshot': 'Belum dipetakan', 'category_count': 'Jumlah kategori', 'rare_category_count': 'Kategori langka (<100 snapshot)', 'rare_snapshot_count': 'Snapshot pada kategori langka', 'unmatched_percentage': 'Unmatched (%)', 'rare_snapshot_percentage': 'Kategori langka (%)', 'feature_decision': 'Keputusan'})
display(coverage_display)
coverage_plot = master_coverage.melt(id_vars='feature_name', value_vars=['unmatched_percentage','rare_snapshot_percentage'], var_name='coverage_issue', value_name='percentage')
coverage_plot['coverage_issue'] = coverage_plot.coverage_issue.map({'unmatched_percentage':'Belum cocok master','rare_snapshot_percentage':'Kategori langka'})
plt.figure(figsize=(7, 4)); sns.barplot(data=coverage_plot, x='feature_name', y='percentage', hue='coverage_issue').set(title='Dampak masalah master terhadap snapshot training', xlabel='Fitur', ylabel='Persentase snapshot (%)'); plt.tight_layout(); plt.show()
location_coverage = master_coverage.loc[master_coverage.feature_name.eq('LOCATION')].iloc[0]
display(Markdown(f"Lokasi belum cocok master hanya memengaruhi **{int(location_coverage.unmatched_snapshot):,} dari {int(location_coverage.total_snapshot):,} snapshot ({float(location_coverage.unmatched_percentage):.4f}%)**. Secara coverage, lokasi aman diuji sebagai prediktor, tetapi tetap gunakan kategori `UNKNOWN`, missing flag, minimum support, dan bandingkan model dengan-versus-tanpa lokasi.".replace(',', '.')))
missing = query('SELECT * FROM analytics.eda_feature_missingness ORDER BY missing_percentage DESC')
feature_labels = {'item_model_code_clean': 'Model PART', 'installed_client_clean': 'Client pemasangan', 'last_place_clean': 'Lokasi terakhir', 'days_since_installation': 'Hari sejak pemasangan', 'days_since_last_event': 'Hari sejak kegiatan terakhir', 'days_since_last_failure': 'Hari sejak kerusakan terakhir', 'days_since_last_corrective': 'Hari sejak corrective terakhir', 'days_at_last_location': 'Lama di lokasi terakhir'}
handling_labels = {'EXCLUDE_ROW_IF_MISSING_CORE_IDENTITY':'Keluarkan bila identitas inti kosong', 'UNKNOWN_CATEGORY_PLUS_MISSING_FLAG':'Gunakan kategori UNKNOWN dan penanda kosong', 'UNKNOWN_CATEGORY_PLUS_FLAG_COMPARE_WITHOUT_LOCATION':'UNKNOWN + penanda; bandingkan model tanpa lokasi', 'EXCLUDE_IF_MISSING_CYCLE_START':'Keluarkan bila awal cycle tidak diketahui', 'MEDIAN_BY_MODEL_PLUS_MISSING_FLAG':'Median per model + penanda kosong', 'STRUCTURAL_NO_PRIOR_FAILURE_USE_INDICATOR_AND_SENTINEL':'Bukan error: belum pernah failure; gunakan indikator + sentinel', 'STRUCTURAL_NO_PRIOR_CORRECTIVE_USE_INDICATOR_AND_SENTINEL':'Bukan error: belum pernah corrective; gunakan indikator + sentinel', 'MISSING_FLAG_AND_COMPARE_MODEL_WITHOUT_LOCATION_AGE':'Penanda kosong; bandingkan tanpa umur lokasi'}
missing['missing_percentage'] = pd.to_numeric(missing.missing_percentage, errors='coerce')
missing_display = missing.assign(feature_name=missing['feature_name'].replace(feature_labels), recommended_handling=missing['recommended_handling'].replace(handling_labels)).rename(columns={'feature_name': 'Informasi', 'missing_count': 'Jumlah kosong', 'missing_percentage': 'Persentase kosong', 'available_count': 'Jumlah tersedia', 'recommended_handling': 'Strategi sebelum modeling'})
display(missing_display[['Informasi','Jumlah kosong','Persentase kosong','Jumlah tersedia','Strategi sebelum modeling']])
plt.figure(figsize=(8, 5)); sns.barplot(data=missing_display, y='Informasi', x='Persentase kosong', color='steelblue').set(title='Persentase informasi yang masih kosong', xlabel='Kosong (%)', ylabel=''); plt.tight_layout(); plt.show()

### 3.4 Outlier dan daftar manual review

Outlier adalah tanda pemeriksaan, bukan alasan menghapus otomatis. Setiap temuan
dibaca bersama dampak dan konteks proses bisnisnya.


In [ ]:
outliers = query('SELECT * FROM analytics.eda_outlier_summary ORDER BY affected_count DESC')
outlier_labels = {'OPERATIONAL_GAP_GT_10Y': 'Jarak kegiatan lebih dari 10 tahun', 'ZERO_OR_NEGATIVE_DURATION_CYCLE': 'Cycle tanpa durasi positif', 'FAILURE_NOT_PRECEDED_BY_INSTALLED': 'Failure tidak langsung didahului INSTALLED', 'ITEM_WITH_5_PLUS_FAILURES': 'PART dengan minimal 5 failure', 'JOURNEY_MODEL_INCONSISTENT': 'Model journey tidak konsisten', 'INVALID_OR_FUTURE_JOURNEY_DATE': 'Tanggal journey invalid/masa depan', 'SNAPSHOT_WITHOUT_MASTER_LOCATION': 'Snapshot tanpa lokasi master'}
outliers_display = outliers.assign(check_name=outliers['check_name'].replace(outlier_labels)).rename(columns={'check_name': 'Yang perlu diperiksa', 'affected_count': 'Jumlah terdampak', 'explanation': 'Alasan'})
display(outliers_display)
display(Markdown("""**Interpretasi:** jumlah besar belum otomatis menunjukkan error; gap lama, failure tanpa previous INSTALLED, dan item berulang perlu sampling berbasis kasus.

**Keputusan:** pertahankan untuk audit, lalu keluarkan hanya melalui quality rule yang sudah disetujui."""))


In [ ]:
cycle_quality = query("""SELECT cycle_quality_status, COUNT(*) cycle_count FROM analytics.item_installation_cycle GROUP BY 1 ORDER BY 2 DESC""")
snapshot_quality = query("""SELECT target_quality_status, COUNT(*) snapshot_count FROM analytics.item_observation_30d GROUP BY 1 ORDER BY 2 DESC""")
display(cycle_quality.rename(columns={'cycle_quality_status':'Kualitas cycle','cycle_count':'Jumlah cycle'}))
display(snapshot_quality.rename(columns={'target_quality_status':'Keputusan label snapshot','snapshot_count':'Jumlah snapshot'}))
unknown_reinstall_count = int(cycle_quality.loc[cycle_quality.cycle_quality_status.eq('UNKNOWN_REINSTALL_WITHOUT_RECORDED_FAILURE'),'cycle_count'].sum())
coverage_unconfirmed_count = int(cycle_quality.loc[cycle_quality.cycle_quality_status.eq('RIGHT_CENSORED_ACTIVITY_COVERAGE_UNCONFIRMED'),'cycle_count'].sum())
display(Markdown(f"""**Temuan utama:** **{unknown_reinstall_count:,} cycle reinstall** tidak lagi otomatis menjadi negatif. **{coverage_unconfirmed_count:,} cycle right-censored** memiliki coverage aktivitas yang belum terkonfirmasi dan disediakan sebagai sensitivity flag.

**Keputusan:** failure positif tidak diubah; label negatif dibuat lebih konservatif dan dapat diaudit.""".replace(',', '.')))


## 4. Target dan kualitas label

**Pertanyaan:** seberapa tidak seimbang target, bagaimana karakteristik
snapshot positif dan negatif, dan seberapa bisa dipercaya label yang dipakai
untuk training? Bagian ini murni tentang kesiapan label untuk modeling, bukan
insight bisnis (lihat notebook `01a_business_eda.ipynb` untuk insight risiko
per model/lokasi/klien).


### 4.1 Distribusi target dan imbalance

Target positif berarti failure terjadi setelah snapshot dan maksimal 30 hari.
Negatif hanya digunakan jika follow-up penuh tersedia dan cycle tidak berakhir
reinstall tanpa failure tercatat.


In [ ]:
target_distribution = query('SELECT * FROM analytics.eda_target_class_distribution ORDER BY label_value')
target_display = target_distribution.rename(columns={'label_value': 'Label', 'label_name': 'Arti label', 'snapshot_count': 'Jumlah snapshot', 'class_percentage': 'Persentase (%)', 'negative_to_positive_ratio': 'Rasio negatif : positif', 'imbalance_status': 'Status imbalance'})
display(target_display)
positive_row = target_distribution.loc[target_distribution.label_value.eq(1)].iloc[0]
display(Markdown(f"**Kesimpulan imbalance:** hanya **{float(positive_row.class_percentage):.4f}%** snapshot yang positif. Terdapat sekitar **{float(positive_row.negative_to_positive_ratio):.2f} snapshot negatif untuk setiap 1 snapshot positif**. Karena itu evaluasi model nanti wajib memakai precision, recall, PR-AUC, ROC-AUC, calibration, dan confusion matrix; akurasi tidak boleh dipakai sendirian."))
plt.figure(figsize=(7, 4)); ax=sns.barplot(data=target_distribution, x='label_name', y='snapshot_count', hue='label_name', legend=False); ax.set_yscale('log'); ax.set(title='Jumlah snapshot per label (skala log)', xlabel='', ylabel='Jumlah snapshot'); plt.xticks(rotation=10); plt.tight_layout(); plt.show()


### 4.2 Karakteristik snapshot positif dan negatif

Perbandingan memakai fitur yang tersedia pada tanggal snapshot. Sampel negatif
diambil deterministik agar kelompok positif yang langka tetap terlihat.


In [ ]:
sample = query("""SELECT target_failure_30d, days_since_installation, prior_failure_count, prior_corrective_count, prior_relocation_count, prior_distinct_places FROM analytics.item_observation_30d WHERE is_training_eligible AND (target_failure_30d OR MOD(ABS(HASHTEXT(installation_cycle_id || observation_date::text)::bigint), 100) < 3)""")
numeric_features = ['days_since_installation', 'prior_failure_count', 'prior_corrective_count', 'prior_relocation_count', 'prior_distinct_places']
sample[numeric_features] = sample[numeric_features].apply(pd.to_numeric, errors='coerce')
feature_summary = sample.groupby('target_failure_30d').agg(observations=('days_since_installation', 'size'), mean_age_days=('days_since_installation', 'mean'), median_age_days=('days_since_installation', 'median'), avg_prior_failures=('prior_failure_count', 'mean'), avg_prior_corrective=('prior_corrective_count', 'mean'), avg_prior_relocations=('prior_relocation_count', 'mean'), avg_prior_places=('prior_distinct_places', 'mean')).round(2)
feature_summary.index = feature_summary.index.map({False: 'Tidak rusak dalam 30 hari', True: 'Rusak dalam 30 hari'})
display(feature_summary.rename(columns={'observations': 'Jumlah snapshot', 'mean_age_days': 'Rata-rata umur (hari)', 'median_age_days': 'Median umur (hari)', 'avg_prior_failures': 'Rata-rata kerusakan sebelumnya', 'avg_prior_corrective': 'Rata-rata corrective sebelumnya', 'avg_prior_relocations': 'Rata-rata perpindahan', 'avg_prior_places': 'Rata-rata jumlah lokasi'}))
plot_data = sample[sample.days_since_installation <= sample.days_since_installation.quantile(.99)].copy()
sns.boxplot(data=plot_data, x='target_failure_30d', y='days_since_installation', showfliers=False).set(title='Perbandingan umur PART berdasarkan hasil 30 hari', xlabel='Mengalami kerusakan dalam 30 hari', ylabel='Hari sejak dipasang'); plt.show()

### 4.3 Label gap: RETURNED/outcome tanpa onset tepercaya

RETURNED dan status outcome tidak otomatis membuka failure baru. Tabel ini
mencari kandidat review agar ground truth tetap ketat.


In [ ]:
label_gap_summary = query("""WITH candidate AS (SELECT c.installation_cycle_id, BOOL_OR(o.event_semantic = 'RETURN_FLOW') has_return, BOOL_OR(o.event_semantic = 'FAILURE_OUTCOME') has_failure_outcome FROM analytics.item_installation_cycle c JOIN analytics.item_journey_operational_timeline o ON o.item_identifier_clean = c.item_identifier_clean AND o.created_on > c.installed_on AND o.created_on <= c.cycle_end_on WHERE NOT c.has_observed_failure AND o.event_semantic IN ('RETURN_FLOW', 'FAILURE_OUTCOME') GROUP BY c.installation_cycle_id) SELECT COUNT(*) candidate_cycles, COUNT(*) FILTER (WHERE has_return) returned_cycles, COUNT(*) FILTER (WHERE has_failure_outcome) explicit_failure_outcome_cycles FROM candidate""")
display(label_gap_summary.rename(columns={'candidate_cycles': 'Siklus yang perlu diperiksa', 'returned_cycles': 'Siklus dengan RETURNED', 'explicit_failure_outcome_cycles': 'Siklus dengan status rusak yang jelas'}))
label_gap_detail = query("""SELECT o.event_semantic, o.status_clean, COUNT(*) event_count, COUNT(DISTINCT c.installation_cycle_id) cycle_count FROM analytics.item_installation_cycle c JOIN analytics.item_journey_operational_timeline o ON o.item_identifier_clean = c.item_identifier_clean AND o.created_on > c.installed_on AND o.created_on <= c.cycle_end_on WHERE NOT c.has_observed_failure AND o.event_semantic IN ('RETURN_FLOW', 'FAILURE_OUTCOME') GROUP BY o.event_semantic, o.status_clean ORDER BY o.event_semantic, event_count DESC""")
display(label_gap_detail.rename(columns={'event_semantic': 'Kelompok kejadian', 'status_clean': 'Status', 'event_count': 'Jumlah kejadian', 'cycle_count': 'Jumlah siklus'}))
sns.barplot(data=label_gap_detail, y='status_clean', x='cycle_count', hue='event_semantic').set(title='Status lanjutan tanpa catatan corrective dismantle', xlabel='Jumlah siklus', ylabel='Status'); plt.show()

### 4.4 Failure dengan flow lanjutan belum lengkap

Failure tetap positif bila onset-nya valid. Umur follow-up dipakai untuk
membedakan proses yang mungkin masih berjalan dari kemungkinan histori hilang.


In [ ]:
incomplete = query("""SELECT * FROM analytics.eda_incomplete_failure_summary ORDER BY CASE followup_review_group WHEN 'LIKELY_ONGOING_0_30D' THEN 1 WHEN 'REVIEW_31_180D' THEN 2 ELSE 3 END""")
incomplete_labels = {'LIKELY_ONGOING_0_30D': 'Kemungkinan masih berjalan (0-30 hari)', 'REVIEW_31_180D': 'Perlu dipantau (31-180 hari)', 'LIKELY_HISTORY_GAP_GT_180D': 'Kemungkinan histori hilang (>180 hari)'}
incomplete_display = incomplete.assign(followup_review_group=incomplete['followup_review_group'].replace(incomplete_labels)).rename(columns={'followup_review_group': 'Kelompok pemeriksaan', 'failure_count': 'Jumlah failure', 'item_count': 'Jumlah PART', 'earliest_failure_date': 'Tanggal paling awal', 'latest_failure_date': 'Tanggal paling akhir'})
display(incomplete_display)
sns.barplot(data=incomplete_display, y='Kelompok pemeriksaan', x='Jumlah failure', color='darkorange').set(title='Failure tanpa catatan proses lanjutan', xlabel='Jumlah failure', ylabel=''); plt.show()
missing_onset = query('SELECT item_model_code_clean, installed_on::date installed_on, installed_place_clean, outcome_on::date outcome_on, outcome_status, suggested_label FROM analytics.failure_outcome_missing_onset_review ORDER BY outcome_on')
display(missing_onset.rename(columns={'item_model_code_clean': 'Model PART', 'installed_on': 'Tanggal dipasang', 'installed_place_clean': 'Lokasi pemasangan', 'outcome_on': 'Tanggal status rusak', 'outcome_status': 'Status rusak', 'suggested_label': 'Saran label review'}))

### 4.5 Sensitivitas cadence snapshot

Cadence 7 dan 30 hari dibandingkan berdasarkan failure yang tertangkap dan
berapa kali satu failure muncul sebagai peringatan positif.


In [ ]:
cadence = query('SELECT * FROM analytics.eda_snapshot_cadence_comparison ORDER BY cadence_days')
cadence_numeric = ['all_snapshots', 'eligible_snapshots', 'incomplete_followup_snapshots', 'positive_snapshots', 'positive_percentage', 'failure_cycles', 'captured_failure_cycles', 'uncaptured_failure_cycles', 'average_positive_snapshots_per_failure']
cadence[cadence_numeric] = cadence[cadence_numeric].apply(pd.to_numeric, errors='coerce')
display(cadence.rename(columns={'cadence_days': 'Jarak snapshot (hari)', 'all_snapshots': 'Semua snapshot', 'eligible_snapshots': 'Layak digunakan', 'incomplete_followup_snapshots': 'Follow-up belum lengkap', 'positive_snapshots': 'Snapshot positif', 'positive_percentage': 'Persentase positif', 'failure_cycles': 'Cycle dengan failure', 'captured_failure_cycles': 'Failure tertangkap', 'uncaptured_failure_cycles': 'Failure tidak tertangkap', 'average_positive_snapshots_per_failure': 'Rata-rata peringatan per failure'}))
sns.barplot(data=cadence, x='cadence_days', y='average_positive_snapshots_per_failure', color='steelblue').set(title='Berapa kali satu failure mendapat peringatan?', xlabel='Jarak snapshot (hari)', ylabel='Rata-rata snapshot positif'); plt.show()
unit_comparison = query('SELECT * FROM analytics.eda_failure_unit_comparison ORDER BY analysis_unit')
display(unit_comparison.rename(columns={'analysis_unit': 'Cara menghitung', 'population_count': 'Jumlah yang diperiksa', 'positive_count': 'Jumlah positif', 'positive_percentage': 'Persentase positif', 'explanation': 'Arti'}))
fast_failure = query("""SELECT COUNT(*) FILTER (WHERE failure_onset_on - installed_on <= INTERVAL '1 day') failure_le_1d, COUNT(*) FILTER (WHERE failure_onset_on > installed_on AND failure_onset_on - installed_on <= INTERVAL '1 day') captured_from_first_snapshot FROM analytics.item_installation_cycle WHERE is_initial_model_cohort AND has_observed_failure""")
display(Markdown(f"Ada **{int(fast_failure.failure_le_1d.iloc[0])}** failure dalam maksimal satu hari setelah pemasangan, dan **{int(fast_failure.captured_from_first_snapshot.iloc[0])}** semuanya tertangkap dari snapshot pertama."))

## 5. Hierarki PART-TERMINAL, analisis bivariat, dan multivariat

**Pertanyaan:** apakah PART terhubung dengan TERMINAL, apakah konteks terminal berasosiasi dengan failure PART, dan apakah asosiasi tersebut tetap terlihat setelah karakteristik PART dikontrol?

Relasi bersifat **many-to-one pada satu installation cycle** dan dapat menjadi many-to-many sepanjang histori karena PART dapat berpindah terminal. Mapping menggunakan `host_serial_code + wo_code` installation ke `t_item_request_out`, lalu `parent_serial_code` menjadi identitas TERMINAL. Timestamp pencatatan link tetap diaudit karena relasi historis lama dapat merupakan backfill.


### 5.1 Struktur relasi dan analisis bivariat

Bivariat membandingkan satu fitur dengan target tanpa adjustment. Positive rate, risk ratio, Wilson 95%, minimum support, dan Cramér's V dilaporkan bersama. Hasil ini adalah asosiasi prediktif, bukan sebab-akibat.


In [ ]:
hierarchy_summary = query('SELECT * FROM analytics.eda_part_terminal_structure_summary ORDER BY metric')
hierarchy_summary['value'] = pd.to_numeric(hierarchy_summary.value, errors='coerce')
display(hierarchy_summary.rename(columns={'metric':'Metrik relasi','value':'Jumlah','interpretation':'Interpretasi'}))
association_summary = query('SELECT * FROM analytics.eda_bivariate_association_summary ORDER BY association')
for col in ['chi_square','cramers_v','observation_count']: association_summary[col] = pd.to_numeric(association_summary[col], errors='coerce')
display(association_summary.rename(columns={'association':'Hubungan','chi_square':'Chi-square','cramers_v':"Cramér's V",'observation_count':'N','interpretation':'Interpretasi'}))
terminal_type_risk = query('SELECT * FROM analytics.eda_bivariate_terminal_type_target ORDER BY positive_percentage DESC')
terminal_model_risk = query('SELECT * FROM analytics.eda_bivariate_terminal_model_target WHERE meets_minimum_support ORDER BY positive_percentage DESC')
for frame in [terminal_type_risk, terminal_model_risk]:
    for col in ['snapshot_count','part_count','terminal_count','positive_count','positive_percentage','risk_ratio_to_overall']:
        frame[col] = pd.to_numeric(frame[col], errors='coerce')
display(terminal_type_risk.rename(columns={'terminal_type':'Tipe TERMINAL','snapshot_count':'Snapshot','part_count':'PART','terminal_count':'TERMINAL','positive_count':'Positif','positive_percentage':'Positive rate (%)','risk_ratio_to_overall':'Risk ratio vs overall','wilson_lower_95_pct':'Wilson lower 95%','wilson_upper_95_pct':'Wilson upper 95%','meets_minimum_support':'Minimum support'}))
display(terminal_model_risk.rename(columns={'terminal_model_code':'Model TERMINAL','terminal_type':'Tipe TERMINAL','snapshot_count':'Snapshot','part_count':'PART','terminal_count':'TERMINAL','positive_count':'Positif','positive_percentage':'Positive rate (%)','risk_ratio_to_overall':'Risk ratio vs overall'}))
plot_type = terminal_type_risk.loc[terminal_type_risk.meets_minimum_support & terminal_type_risk.terminal_type.ne('UNMAPPED')].sort_values('positive_percentage', ascending=False)
plt.figure(figsize=(9,4)); sns.barplot(data=plot_type, x='terminal_type', y='positive_percentage', color='darkcyan'); plt.axhline(float(positive_row.class_percentage), color='crimson', linestyle='--', label='Rata-rata seluruh snapshot'); plt.title('Positive rate failure 30 hari menurut tipe TERMINAL'); plt.xlabel('Tipe TERMINAL'); plt.ylabel('Positive rate (%)'); plt.xticks(rotation=25, ha='right'); plt.legend(); plt.tight_layout(); plt.show()
terminal_target_v = float(association_summary.loc[association_summary.association.eq('TERMINAL_TYPE_VS_TARGET'),'cramers_v'].iloc[0])
part_terminal_v = float(association_summary.loc[association_summary.association.eq('PART_MODEL_VS_TERMINAL_TYPE'),'cramers_v'].iloc[0])
display(Markdown(f"""**Temuan bivariat:** tipe TERMINAL berasosiasi dengan target dengan Cramér's V **{terminal_target_v:.4f}**, sedangkan hubungan model PART dengan tipe TERMINAL jauh lebih kuat, V **{part_terminal_v:.4f}**. Karena itu perbedaan rate terminal berpotensi kuat dipengaruhi komposisi model PART dan wajib diuji multivariat."""))

### 5.2 Analisis multivariat bertingkat

Tiga Logistic Regression L2 dibandingkan: **M1 PART-only**, **M2 PART+TERMINAL**, dan **M3 adjusted operational history**. Data 2014-2024 menjadi train, 2025 validation, dan 2026 test; embargo 30 hari tetap berlaku. Seluruh positif dipertahankan dan negatif diambil deterministik 1:25 dengan bobot 25 untuk mengembalikan distribusi populasi. Analisis ini adalah screening multivariat EDA; final modeling tetap memerlukan tuning, calibration, clustered uncertainty, dan sensitivity terhadap link backfill.


In [ ]:
multivariate_sample = query("""
SELECT CASE
         WHEN observation_on + INTERVAL '30 days' < DATE '2025-01-01' THEN 'TRAIN_2014_2024'
         WHEN observation_on >= DATE '2025-01-01' AND observation_on + INTERVAL '30 days' < DATE '2026-01-01' THEN 'VALIDATION_2025'
         WHEN observation_on >= DATE '2026-01-01' THEN 'TEST_2026'
       END split, target_failure_30d::integer target,
       item_model_code_clean,
       CASE WHEN is_parent_link_valid THEN terminal_type ELSE 'UNMAPPED' END terminal_type,
       installed_client_clean, observation_month::text observation_month,
       days_since_installation, total_prior_events, prior_failure_count,
       prior_corrective_count, prior_distinct_places, days_since_last_event,
       days_since_last_failure, days_since_last_corrective, prior_failure_365d,
       is_parent_link_recorded_after_installation
FROM analytics.eda_item_observation_30d_hierarchy
WHERE is_training_eligible AND observation_on >= DATE '2014-01-01'
  AND (target_failure_30d OR MOD(ABS(HASHTEXT(installation_cycle_id || observation_date::text)::bigint),25)=0)
  AND NOT (observation_on < DATE '2025-01-01' AND observation_on + INTERVAL '30 days' >= DATE '2025-01-01')
  AND NOT (observation_on < DATE '2026-01-01' AND observation_on + INTERVAL '30 days' >= DATE '2026-01-01')
""")
multivariate_sample['sample_weight'] = np.where(multivariate_sample.target.eq(1), 1.0, 25.0)
def _prepare_design(train, frames, categorical, numeric):
    matrices = [np.ones((len(frame),1), dtype=float) for frame in frames]
    names = ['intercept']
    for col in categorical:
        train_value = train[col].astype('string').fillna('UNKNOWN')
        counts = train_value.value_counts()
        levels = list(counts[counts.ge(50)].index)
        reference = 'GATE' if col == 'terminal_type' and 'GATE' in levels else levels[0]
        levels = [reference] + [v for v in levels if v != reference]
        for level in levels[1:] + ['OTHER']:
            for i, frame in enumerate(frames):
                value = frame[col].astype('string').fillna('UNKNOWN')
                mapped = value.where(value.isin(levels), 'OTHER')
                matrices[i] = np.column_stack([matrices[i], mapped.eq(level).to_numpy(float)])
            names.append(f'{col}={level}')
    for col in numeric:
        train_raw = pd.to_numeric(train[col], errors='coerce')
        median = float(train_raw.median()) if train_raw.notna().any() else 0.0
        transformed_train = np.log1p(np.clip(train_raw.fillna(median).to_numpy(float),0,None))
        mean, std = transformed_train.mean(), transformed_train.std() or 1.0
        for i, frame in enumerate(frames):
            raw = pd.to_numeric(frame[col], errors='coerce')
            missing = raw.isna().to_numpy(float)
            value = (np.log1p(np.clip(raw.fillna(median).to_numpy(float),0,None))-mean)/std
            matrices[i] = np.column_stack([matrices[i], value, missing])
        names.extend([col, f'{col}_missing'])
    return matrices, names
def _fit_ridge_logistic(X, y, weight, l2=1.0, max_iter=30):
    beta = np.zeros(X.shape[1]); penalty = np.ones(X.shape[1]); penalty[0] = 0
    weight = np.asarray(weight,float); weight = weight/weight.mean()
    for _ in range(max_iter):
        probability = 1/(1+np.exp(-np.clip(X@beta,-30,30)))
        curvature = weight*probability*(1-probability)
        hessian = X.T@(X*curvature[:,None]) + l2*np.diag(penalty)
        gradient = X.T@(weight*(y-probability)) - l2*penalty*beta
        step = np.linalg.solve(hessian, gradient); beta += step
        if np.max(np.abs(step)) < 1e-6: break
    return beta
def _weighted_metrics(y, probability, weight):
    y=np.asarray(y,int); p=np.clip(np.asarray(probability,float),1e-9,1-1e-9); w=np.asarray(weight,float)
    grouped=pd.DataFrame({'p':p,'y':y,'w':w}).groupby('p',sort=True).apply(lambda g: pd.Series({'pos':(g.y*g.w).sum(),'neg':((1-g.y)*g.w).sum()}),include_groups=False)
    ranked=grouped.sort_index(ascending=False); cumulative_positive=ranked.pos.cumsum(); cumulative_total=(ranked.pos+ranked.neg).cumsum()
    ap=float(np.sum((cumulative_positive/cumulative_total)*(ranked.pos/ranked.pos.sum())))
    cum_neg=grouped.neg.cumsum()-grouped.neg; auc=float((grouped.pos*(cum_neg+.5*grouped.neg)).sum()/(grouped.pos.sum()*grouped.neg.sum()))
    return {'PR-AUC':ap,'ROC-AUC':auc,'Brier':float(np.average((y-p)**2,weights=w)),'Log loss':float(np.average(-(y*np.log(p)+(1-y)*np.log(1-p)),weights=w))}
train_mv=multivariate_sample.loc[multivariate_sample.split.eq('TRAIN_2014_2024')].copy(); validation_mv=multivariate_sample.loc[multivariate_sample.split.eq('VALIDATION_2025')].copy(); test_mv=multivariate_sample.loc[multivariate_sample.split.eq('TEST_2026')].copy()
model_specs={
 'M1 PART-only': (['item_model_code_clean'],['days_since_installation']),
 'M2 PART+TERMINAL': (['item_model_code_clean','terminal_type'],['days_since_installation']),
 'M3 Adjusted history': (['item_model_code_clean','terminal_type','installed_client_clean','observation_month'],['days_since_installation','total_prior_events','prior_failure_count','prior_corrective_count','prior_distinct_places','days_since_last_event','days_since_last_failure','days_since_last_corrective','prior_failure_365d'])}
results=[]; fitted={}
base_probability=np.average(train_mv.target,weights=train_mv.sample_weight)
for split_name,frame in [('Validation 2025',validation_mv),('Test 2026',test_mv)]:
    results.append({'Model':'M0 Prevalence','Periode':split_name,**_weighted_metrics(frame.target,np.full(len(frame),base_probability),frame.sample_weight)})
for model_name,(categorical,numeric) in model_specs.items():
    (x_train,x_validation,x_test),feature_names=_prepare_design(train_mv,[train_mv,validation_mv,test_mv],categorical,numeric)
    coefficient=_fit_ridge_logistic(x_train,train_mv.target.to_numpy(),train_mv.sample_weight.to_numpy())
    fitted[model_name]=(coefficient,feature_names)
    for split_name,frame,X in [('Validation 2025',validation_mv,x_validation),('Test 2026',test_mv,x_test)]:
        probability=1/(1+np.exp(-np.clip(X@coefficient,-30,30)))
        results.append({'Model':model_name,'Periode':split_name,**_weighted_metrics(frame.target,probability,frame.sample_weight)})
multivariate_result=pd.DataFrame(results)
display(multivariate_result.round(5))
coef,names=fitted['M3 Adjusted history']; adjusted_terminal=pd.DataFrame({'Fitur':names,'Koefisien':coef}); adjusted_terminal=adjusted_terminal[adjusted_terminal.Fitur.str.startswith('terminal_type=')].copy(); adjusted_terminal['Adjusted odds multiplier']=np.exp(adjusted_terminal.Koefisien); display(adjusted_terminal.sort_values('Adjusted odds multiplier',ascending=False).round(4))
pivot_pr=multivariate_result.pivot(index='Model',columns='Periode',values='PR-AUC'); pivot_pr.plot(kind='bar',figsize=(10,4),color=['steelblue','darkorange']); plt.title('Perbandingan PR-AUC multivariat secara temporal'); plt.ylabel('PR-AUC berbobot'); plt.xticks(rotation=20,ha='right'); plt.tight_layout(); plt.show()
display(Markdown("**Cara membaca:** kenaikan M1→M2 mengukur nilai tambah konteks TERMINAL setelah model PART dan umur dikontrol. M2→M3 menunjukkan nilai tambah histori operasional. Koefisien terminal adalah odds multiplier teradjust regularized terhadap referensi GATE, bukan estimasi kausal atau confidence interval final. Kategori dengan overlap model PART yang lemah—terutama Balance Reader yang hanya didukung satu model PART—tidak mempunyai efek terminal independen yang teridentifikasi dengan baik."))

## 6. Feature readiness dan transformasi sebelum modeling

**Pertanyaan:** fitur mana yang tersedia, redundan, berpotensi informatif, atau
berisiko leakage? Feature selection hanya boleh belajar dari periode train.
Karena itu split waktu ditentukan lebih dahulu, lalu korelasi dan IV dihitung
pada train 2014-2024 dengan embargo 30 hari.


### 6.1 Time-based split dan embargo target

Random split dilarang. Snapshot yang horizon 30 harinya menyeberangi batas tahun
train/validation dikeluarkan sebagai embargo. Data 2026 tidak digunakan untuk
memilih fitur, parameter, atau threshold.


In [ ]:
splits = query("""
SELECT split, COUNT(*) observations, COUNT(DISTINCT item_identifier_clean) items,
       COUNT(*) FILTER (WHERE target_failure_30d) positives,
       ROUND(100.0*COUNT(*) FILTER (WHERE target_failure_30d)/NULLIF(COUNT(*),0),4) positive_pct
FROM (
    SELECT o.*,
        CASE
          WHEN observation_on < DATE '2014-01-01' THEN 'EXCLUDED_PRE_2014'
          WHEN observation_on + INTERVAL '30 days' < DATE '2025-01-01' THEN 'TRAIN_2014_2024'
          WHEN observation_on < DATE '2025-01-01' THEN 'EXCLUDED_TRAIN_EMBARGO'
          WHEN observation_on + INTERVAL '30 days' < DATE '2026-01-01' THEN 'VALIDATION_2025'
          WHEN observation_on < DATE '2026-01-01' THEN 'EXCLUDED_VALIDATION_EMBARGO'
          ELSE 'TEST_2026'
        END split
    FROM analytics.item_observation_30d o
    WHERE is_training_eligible
) s GROUP BY split ORDER BY MIN(observation_on)
""")
display(splits.rename(columns={'split':'Pembagian waktu','observations':'Snapshot','items':'PART','positives':'Positif','positive_pct':'Positive rate (%)'}))
display(Markdown("""**Keputusan:** gunakan train 2014-2024, validation 2025, dan test 2026. Baris embargo/pre-2014 tidak masuk training baseline utama; data 2013 hanya digunakan untuk sensitivity comparison."""))


### 6.2 Missing, korelasi, redundansi, dan Information Value

Pearson dan Spearman memeriksa redundancy; IV adalah screening univariat, bukan
feature importance final. Seluruh perhitungan pemilihan fitur di bagian ini
dibatasi pada train agar validation/test tidak bocor ke keputusan fitur.


In [ ]:
candidate_numeric_features = ['days_since_installation','days_since_last_event','days_since_last_failure','days_since_last_corrective','days_at_last_location','total_prior_events','prior_failure_count','prior_corrective_count','prior_relocation_count','prior_preventive_count','prior_repair_process_count','prior_events_30d','prior_events_90d','prior_events_180d','prior_corrective_30d','prior_corrective_90d','prior_corrective_180d','prior_preventive_90d','prior_failure_365d','prior_distinct_places']
feature_sql = ', '.join(candidate_numeric_features)
correlation_sample = query(f"SELECT target_failure_30d, {feature_sql} FROM analytics.item_observation_30d WHERE is_training_eligible AND observation_on >= DATE '2014-01-01' AND observation_on + INTERVAL '30 days' < DATE '2025-01-01' AND MOD(ABS(HASHTEXT(installation_cycle_id || observation_date::text)::bigint), 20)=0")
correlation_sample[candidate_numeric_features] = correlation_sample[candidate_numeric_features].apply(pd.to_numeric, errors='coerce')
correlation_features = [c for c in candidate_numeric_features if correlation_sample[c].nunique(dropna=True) > 1]
pearson_corr = correlation_sample[correlation_features].corr(method='pearson')
spearman_corr = correlation_sample[correlation_features].corr(method='spearman')
fig, axes = plt.subplots(1, 2, figsize=(22, 9)); sns.heatmap(pearson_corr, cmap='coolwarm', center=0, vmin=-1, vmax=1, ax=axes[0]); axes[0].set_title('Korelasi Pearson (hubungan linear)'); sns.heatmap(spearman_corr, cmap='coolwarm', center=0, vmin=-1, vmax=1, ax=axes[1]); axes[1].set_title('Korelasi Spearman (urutan/monotonik)'); plt.tight_layout(); plt.show()
upper_mask = np.triu(np.ones(spearman_corr.shape, dtype=bool), k=1)
redundant_pairs = spearman_corr.abs().where(upper_mask).stack().reset_index()
redundant_pairs.columns = ['feature_1','feature_2','abs_spearman']
redundant_pairs = redundant_pairs.loc[redundant_pairs.abs_spearman.ge(0.80)].sort_values('abs_spearman', ascending=False)
redundant_pairs['keputusan'] = np.where(redundant_pairs.abs_spearman.ge(0.95), 'Hampir duplikat: prioritaskan satu setelah validasi waktu', 'Redundan tinggi: uji salah satu/interaksi, jangan masukkan buta')
display(redundant_pairs.rename(columns={'feature_1':'Fitur pertama','feature_2':'Fitur kedua','abs_spearman':'|Spearman|','keputusan':'Tindakan'}))
iv_sample = query(f"SELECT target_failure_30d, {feature_sql} FROM analytics.item_observation_30d WHERE is_training_eligible AND observation_on >= DATE '2014-01-01' AND observation_on + INTERVAL '30 days' < DATE '2025-01-01' AND (target_failure_30d OR MOD(ABS(HASHTEXT(installation_cycle_id || observation_date::text)::bigint),100)<3)")
iv_sample[candidate_numeric_features] = iv_sample[candidate_numeric_features].apply(pd.to_numeric, errors='coerce')
def calculate_information_value(frame, feature, target='target_failure_30d'):
    work = frame[[feature, target]].copy()
    nonmissing = work[feature].dropna()
    if nonmissing.nunique() > 10:
        bins = pd.Series('MISSING', index=work.index, dtype='object')
        try:
            bins.loc[nonmissing.index] = pd.qcut(nonmissing, q=10, duplicates='drop').astype(str)
        except ValueError:
            bins.loc[nonmissing.index] = nonmissing.astype(str)
    else:
        bins = work[feature].astype('string').fillna('MISSING')
    table = pd.crosstab(bins, work[target].astype(bool))
    negative = table.get(False, pd.Series(0, index=table.index)).astype(float) + 0.5
    positive = table.get(True, pd.Series(0, index=table.index)).astype(float) + 0.5
    negative_dist, positive_dist = negative / negative.sum(), positive / positive.sum()
    return float(((positive_dist-negative_dist) * np.log(positive_dist/negative_dist)).sum())
iv_result = pd.DataFrame({'feature': candidate_numeric_features, 'information_value': [calculate_information_value(iv_sample, f) for f in candidate_numeric_features]})
iv_result['screening'] = pd.cut(iv_result.information_value, bins=[-np.inf,.02,.10,.30,.50,np.inf], labels=['Sangat lemah','Lemah','Sedang','Kuat','Sangat kuat: cek leakage/drift'])
iv_result = iv_result.sort_values('information_value', ascending=False)
display(iv_result.rename(columns={'feature':'Fitur','information_value':'Information Value (IV)','screening':'Interpretasi awal'}))
plt.figure(figsize=(9, 7)); sns.barplot(data=iv_result, y='feature', x='information_value', color='darkcyan').set(title='Predictive power awal per fitur (IV univariat)', xlabel='Information Value', ylabel='Fitur'); plt.tight_layout(); plt.show()
display(Markdown(f"""**Temuan utama:** terdapat **{len(redundant_pairs)} pasangan fitur** dengan |Spearman| minimal 0,80. IV train tertinggi adalah **{iv_result.iloc[0].feature} ({iv_result.iloc[0].information_value:.4f})**.

**Tindak lanjut:** bandingkan feature subset melalui temporal validation; IV tinggi wajib diaudit terhadap missing struktural, leakage, dan drift."""))


### 6.3 Keputusan final fitur dan output feature engineering

Keputusan fitur dibakukan dalam catalog database agar alasan keep/drop tidak hanya berada di narasi notebook. `failure_30d_baseline_features` hanya berisi key dan fitur baseline yang telah ditransformasi; `failure_30d_challenger_features` menambahkan lokasi, hierarchy terminal, interaksi, serta kandidat redundan untuk eksperimen terkontrol. Label dan temporal split berada terpisah pada `failure_30d_model_labels`. View `failure_30d_model_audit` hanya untuk QA dan tidak boleh menjadi input training langsung.


In [ ]:
feature_catalog = query('SELECT * FROM analytics.failure_30d_feature_catalog ORDER BY decision, feature_group, feature_name')
feature_quality = query('SELECT * FROM analytics.failure_30d_feature_quality_summary ORDER BY metric')
feature_split = query("SELECT temporal_split, COUNT(*) snapshots, COUNT(*) FILTER (WHERE target_failure_30d) positives FROM analytics.failure_30d_model_labels GROUP BY temporal_split ORDER BY temporal_split")
decision_summary = feature_catalog.groupby('decision',as_index=False).size().rename(columns={'decision':'Keputusan','size':'Jumlah fitur'})
display(decision_summary)
display(feature_catalog.loc[feature_catalog.decision.str.startswith('KEEP')].rename(columns={'feature_name':'Fitur hasil','source_column':'Sumber','feature_group':'Kelompok','decision':'Keputusan','transformation':'Transformasi','missing_handling':'Penanganan missing','rationale':'Alasan'}))
display(feature_catalog.loc[feature_catalog.decision.str.startswith('DROP')].rename(columns={'feature_name':'Fitur dibuang','source_column':'Sumber','feature_group':'Kelompok','decision':'Keputusan','transformation':'Transformasi','missing_handling':'Penanganan missing','rationale':'Alasan'}))
display(feature_quality.rename(columns={'metric':'Quality check','value':'Nilai','expectation':'Ekspektasi'}))
display(feature_split.rename(columns={'temporal_split':'Split waktu','snapshots':'Snapshot','positives':'Positif'}))
engineered_sample = query("SELECT * FROM analytics.failure_30d_baseline_features ORDER BY observation_on DESC LIMIT 10")
display(engineered_sample)
baseline_kept = int(feature_catalog.decision.eq('KEEP_BASELINE').sum()); challenger_kept = int(feature_catalog.decision.eq('KEEP_CHALLENGER').sum()); dropped = int(feature_catalog.decision.str.startswith('DROP').sum())
display(Markdown(f"""**Output siap modeling:** **{baseline_kept} fitur baseline**, **{challenger_kept} kandidat challenger**, dan **{dropped} fitur/kelompok raw yang dibuang atau diganti transformasi**. Quality gate memastikan jumlah feature row sama dengan label, key ganda nol, seluruh fitur inti/non-null hasil engineering lengkap, dan tidak ada nama kolom target/future pada feature views.

Join training yang diwajibkan: `(installation_cycle_id, item_identifier_clean, observation_on)`, filter `temporal_split`, lalu pilih view baseline atau challenger—jangan membaca `failure_30d_model_audit` sebagai matriks training."""))

## 7. Stability dan drift fitur

**Pertanyaan:** apakah pola dan fitur stabil setelah periode train? PSI
2025/2026 dibandingkan terhadap 2024, lalu stabilitas risiko per model dan
lokasi diperiksa. Tahun 2026 masih parsial sampai cutoff data.


### 7.1 Feature drift dan stability model/lokasi

PSI <0,10 relatif stabil, 0,10-0,25 perlu dipantau, dan >=0,25 menunjukkan
drift besar. Stability risiko tetap dilihat per model dan lokasi.


In [ ]:
monthly_feature_stability = query("SELECT * FROM analytics.eda_feature_stability_monthly WHERE observation_month_start >= DATE '2024-01-01' ORDER BY feature_name, observation_month_start")
monthly_feature_stability['observation_month_start'] = pd.to_datetime(monthly_feature_stability.observation_month_start)
for col in ['mean_value','median_value','p90_value','missing_percentage']: monthly_feature_stability[col] = pd.to_numeric(monthly_feature_stability[col], errors='coerce')
monthly_plot = monthly_feature_stability[monthly_feature_stability.feature_name.isin(['prior_events_90d','prior_corrective_90d','prior_failure_365d','days_since_installation'])]
g = sns.relplot(data=monthly_plot, x='observation_month_start', y='mean_value', col='feature_name', col_wrap=2, kind='line', marker='o', facet_kws={'sharey':False}, height=3.4, aspect=1.6); g.set_axis_labels('Bulan snapshot','Rata-rata fitur'); g.set_titles('{col_name}'); g.fig.suptitle('Perubahan rata-rata fitur utama per bulan', y=1.03); plt.show()
drift_sample = query(f"SELECT observation_year, {feature_sql} FROM analytics.item_observation_30d WHERE is_training_eligible AND observation_on>=DATE '2024-01-01' AND MOD(ABS(HASHTEXT(installation_cycle_id || observation_date::text)::bigint),10)=0")
drift_sample[candidate_numeric_features] = drift_sample[candidate_numeric_features].apply(pd.to_numeric, errors='coerce')
def population_stability_index(reference, current):
    reference, current = pd.Series(reference), pd.Series(current)
    if reference.dropna().nunique() <= 10:
        ref_group = reference.astype('object').where(reference.notna(), 'MISSING').astype(str)
        cur_group = current.astype('object').where(current.notna(), 'MISSING').astype(str)
    else:
        internal = np.unique(reference.dropna().quantile(np.linspace(0,1,11)).to_numpy())[1:-1]
        edges = np.r_[-np.inf, internal, np.inf]
        ref_group = pd.cut(reference, edges, include_lowest=True, duplicates='drop').astype('string').fillna('MISSING')
        cur_group = pd.cut(current, edges, include_lowest=True, duplicates='drop').astype('string').fillna('MISSING')
    categories = sorted(set(ref_group) | set(cur_group))
    ref_pct = ref_group.value_counts(normalize=True).reindex(categories, fill_value=0).to_numpy(float)
    cur_pct = cur_group.value_counts(normalize=True).reindex(categories, fill_value=0).to_numpy(float)
    ref_pct, cur_pct = np.clip(ref_pct, 1e-4, None), np.clip(cur_pct, 1e-4, None)
    ref_pct, cur_pct = ref_pct/ref_pct.sum(), cur_pct/cur_pct.sum()
    return float(np.sum((cur_pct-ref_pct)*np.log(cur_pct/ref_pct)))
psi_rows = []
for feature in candidate_numeric_features:
    reference = drift_sample.loc[drift_sample.observation_year.eq(2024), feature]
    for year in [2025, 2026]:
        current = drift_sample.loc[drift_sample.observation_year.eq(year), feature]
        psi_rows.append((feature, year, population_stability_index(reference, current)))
psi_result = pd.DataFrame(psi_rows, columns=['feature','comparison_year','psi'])
psi_result['drift_status'] = pd.cut(psi_result.psi, bins=[-np.inf,.10,.25,np.inf], labels=['Relatif stabil','Perlu dipantau','Drift besar'])
display(psi_result.sort_values(['comparison_year','psi'], ascending=[True,False]).rename(columns={'feature':'Fitur','comparison_year':'Tahun dibanding 2024','psi':'PSI','drift_status':'Status drift'}))
psi_heatmap = psi_result.pivot(index='feature', columns='comparison_year', values='psi')
plt.figure(figsize=(7, 8)); sns.heatmap(psi_heatmap, annot=True, fmt='.3f', cmap='YlOrRd', vmin=0, cbar_kws={'label':'PSI terhadap 2024'}); plt.title('Drift distribusi fitur terhadap referensi 2024'); plt.xlabel('Tahun'); plt.ylabel('Fitur'); plt.tight_layout(); plt.show()
model_stability = query("""WITH top_model AS (SELECT item_model_code_clean FROM analytics.item_observation_30d WHERE is_training_eligible GROUP BY 1 ORDER BY COUNT(*) FILTER (WHERE target_failure_30d) DESC LIMIT 10) SELECT CASE WHEN observation_on < DATE '2025-01-01' THEN '2013-2024' WHEN observation_on < DATE '2026-01-01' THEN '2025' ELSE '2026' END period, item_model_code_clean, COUNT(*) observations, COUNT(DISTINCT item_identifier_clean) items, COUNT(*) FILTER (WHERE target_failure_30d) positives, ROUND(100.0 * COUNT(*) FILTER (WHERE target_failure_30d) / COUNT(*), 4) positive_pct FROM analytics.item_observation_30d JOIN top_model USING (item_model_code_clean) WHERE is_training_eligible GROUP BY 1, 2 HAVING COUNT(DISTINCT item_identifier_clean) >= 20""")
model_stability['positive_pct'] = pd.to_numeric(model_stability['positive_pct'], errors='coerce')
model_stability_heatmap = model_stability.pivot(index='item_model_code_clean', columns='period', values='positive_pct')
plt.figure(figsize=(7, 7)); sns.heatmap(model_stability_heatmap, annot=True, fmt='.2f', cmap='YlOrRd', cbar_kws={'label': 'Persentase positif (%)'}); plt.title('Perubahan risiko model PART menurut periode'); plt.xlabel('Periode'); plt.ylabel('Model PART'); plt.tight_layout(); plt.show()
location_stability = query("""WITH top_location AS (SELECT last_place_clean FROM analytics.item_observation_30d WHERE is_training_eligible AND is_location_feature_eligible GROUP BY 1 ORDER BY COUNT(*) FILTER (WHERE target_failure_30d) DESC LIMIT 10) SELECT CASE WHEN observation_on < DATE '2025-01-01' THEN '2013-2024' WHEN observation_on < DATE '2026-01-01' THEN '2025' ELSE '2026' END period, last_place_clean, COUNT(*) observations, COUNT(DISTINCT item_identifier_clean) items, COUNT(*) FILTER (WHERE target_failure_30d) positives, ROUND(100.0 * COUNT(*) FILTER (WHERE target_failure_30d) / COUNT(*), 4) positive_pct FROM analytics.item_observation_30d JOIN top_location USING (last_place_clean) WHERE is_training_eligible AND is_location_feature_eligible GROUP BY 1, 2 HAVING COUNT(DISTINCT item_identifier_clean) >= 30""")
location_stability['positive_pct'] = pd.to_numeric(location_stability['positive_pct'], errors='coerce')
location_stability_heatmap = location_stability.pivot(index='last_place_clean', columns='period', values='positive_pct')
plt.figure(figsize=(7, 8)); sns.heatmap(location_stability_heatmap, annot=True, fmt='.2f', cmap='YlOrRd', cbar_kws={'label': 'Persentase positif (%)'}); plt.title('Perubahan risiko lokasi menurut periode'); plt.xlabel('Periode'); plt.ylabel('Lokasi terakhir'); plt.tight_layout(); plt.show()
display(Markdown(f"""**Temuan utama:** ada **{int(psi_result.psi.ge(.25).sum())} kombinasi fitur-tahun** dengan PSI minimal 0,25.

**Keputusan:** fitur drift tidak langsung dibuang; lakukan temporal validation, monitoring, dan tentukan kebijakan retraining."""))


### 7.2 Anomali, cleaning decision, dan feature decision

Lonjakan aktivitas, timeline aneh, serta daftar cleaning ditampilkan bersama
tindakan pipeline. Keputusan feature engineering di sini adalah rekomendasi
berdasarkan EDA, belum transformasi modeling final.


In [ ]:
timeline_anomaly = query("""WITH ordered AS (SELECT o.*, MAX(created_on) FILTER (WHERE status_clean='INSTALLED') OVER (PARTITION BY item_identifier_clean ORDER BY created_on, journey_id ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING) prior_installed_on FROM analytics.item_journey_operational_timeline o) SELECT COUNT(*) FILTER (WHERE status_clean='DISMANTLED' AND prior_installed_on IS NULL) dismantle_without_prior_install, COUNT(*) FILTER (WHERE days_since_previous_operational_event<0) negative_gap, COUNT(*) FILTER (WHERE days_since_previous_operational_event=0) zero_gap, COUNT(*) FILTER (WHERE days_since_previous_operational_event>3652.5) gap_gt_10y FROM ordered""")
daily_anomaly_summary = query("SELECT COUNT(*) FILTER (WHERE is_extreme_activity_day) extreme_days, MAX(event_count) max_daily_events, ROUND(AVG(event_count)::numeric,2) average_daily_events, MAX(extreme_event_limit) extreme_limit FROM analytics.eda_daily_activity_anomaly")
display(timeline_anomaly.rename(columns={'dismantle_without_prior_install': 'Dismantle tanpa installation sebelumnya', 'negative_gap': 'Urutan waktu negatif', 'zero_gap': 'Event dengan gap nol', 'gap_gt_10y': 'Gap lebih dari 10 tahun'}))
display(daily_anomaly_summary.rename(columns={'extreme_days': 'Hari dengan lonjakan ekstrem', 'max_daily_events': 'Event terbanyak dalam satu hari', 'average_daily_events': 'Rata-rata event harian', 'extreme_limit': 'Batas outlier harian'}))
top_daily_anomaly = query("SELECT activity_date, event_count, installation_count, dismantle_count, admin_recon_count, bulk_warehouse_reception_count, extreme_event_limit FROM analytics.eda_daily_activity_anomaly WHERE is_extreme_activity_day ORDER BY event_count DESC LIMIT 20")
display(top_daily_anomaly.rename(columns={'activity_date': 'Tanggal', 'event_count': 'Semua event', 'installation_count': 'Installed', 'dismantle_count': 'Dismantled', 'admin_recon_count': 'RECON administratif', 'bulk_warehouse_reception_count': 'Penerimaan gudang massal', 'extreme_event_limit': 'Batas outlier'}))
largest_spike = top_daily_anomaly.iloc[0]
display(Markdown(f"Lonjakan terbesar terjadi pada **{largest_spike.activity_date:%d-%m-%Y}** sebanyak **{int(largest_spike.event_count):,} event**. Rinciannya: **{int(largest_spike.bulk_warehouse_reception_count):,} penerimaan gudang massal**, **{int(largest_spike.installation_count):,} installation**, **{int(largest_spike.dismantle_count):,} dismantle**, dan **{int(largest_spike.admin_recon_count):,} RECON administratif**. Karena itu lonjakan terbesar bukan failure maupun pemasangan.".replace(',', '.')))
daily_activity = query("SELECT activity_date, event_count, is_extreme_activity_day FROM analytics.eda_daily_activity_anomaly ORDER BY activity_date"); daily_activity['activity_date'] = pd.to_datetime(daily_activity.activity_date); daily_activity['event_count'] = pd.to_numeric(daily_activity.event_count)
plt.figure(figsize=(12, 4)); sns.lineplot(data=daily_activity, x='activity_date', y='event_count', color='steelblue', linewidth=.7); extreme_plot=daily_activity[daily_activity.is_extreme_activity_day]; plt.scatter(extreme_plot.activity_date, extreme_plot.event_count, color='crimson', s=12, label='Lonjakan ekstrem'); plt.yscale('symlog'); plt.title('Aktivitas harian dan tanggal dengan lonjakan ekstrem'); plt.xlabel('Tanggal'); plt.ylabel('Jumlah event (skala symlog)'); plt.legend(); plt.tight_layout(); plt.show()
cleaning_actions = query("SELECT suggested_action, COUNT(*) affected_rows FROM analytics.eda_cleaning_review_detail GROUP BY suggested_action ORDER BY affected_rows DESC")
display(cleaning_actions.rename(columns={'suggested_action': 'Tindakan yang disarankan', 'affected_rows': 'Jumlah baris'}))
cleaning_sample = query("SELECT journey_id, item_model_code_clean, status_clean, place_clean, created_on, review_issues::text, suggested_action FROM analytics.eda_cleaning_review_detail ORDER BY suggested_action, journey_id LIMIT 30")
display(cleaning_sample.rename(columns={'journey_id': 'Journey ID', 'item_model_code_clean': 'Model', 'status_clean': 'Status', 'place_clean': 'Lokasi mentah', 'created_on': 'Waktu', 'review_issues': 'Masalah', 'suggested_action': 'Tindakan'}))
cleaning_policy = pd.DataFrame([['RECON administratif','Simpan untuk audit; keluarkan dari perhitungan waktu'],['Penerimaan gudang massal','Simpan sebagai event operasional dengan semantic khusus; jangan artikan sebagai install/dismantle/failure'],['Tanggal invalid/masa depan','Keluarkan dari timeline/model dan periksa sumber'],['Identifier/model inti kosong','Keluarkan dari pembentukan cycle'],['Model tidak konsisten','Keluarkan dari cohort awal model'],['Fuzzy skor tinggi dan margin aman','Gunakan nama canonical; simpan nama sumber, skor, dan metode mapping'],['Fuzzy skor rendah/ambigu','Jangan auto-map; masukkan daftar review'],['Lokasi tidak cocok master','Simpan event; jangan gunakan fitur lokasi'],['Isi log identik','Review journey_id; deduplikasi hanya setelah dikonfirmasi'],['Cycle durasi nol/negatif','Keluarkan dari cohort'],['Snapshot follow-up belum lengkap','Keluarkan dari training'],['Failure tanpa flow lanjutan','Tetap positif jika onset valid; tandai untuk review'],['Reinstall tanpa failure tercatat','Tandai unknown; jangan otomatis menjadi negatif'],['Coverage aktivitas right-censored belum terkonfirmasi','Simpan flag dan uji sensitivity dataset konservatif'],['Cycle masih berjalan','Gunakan hanya snapshot dengan follow-up 30 hari penuh']], columns=['Kondisi','Perlakuan'])
display(cleaning_policy)
calendar_feature_check = query("SELECT COUNT(*) FILTER (WHERE observation_month NOT BETWEEN 1 AND 12) invalid_month, COUNT(*) FILTER (WHERE observation_day_of_week NOT BETWEEN 1 AND 7) invalid_day, COUNT(*) FILTER (WHERE is_weekend IS DISTINCT FROM (observation_day_of_week IN (6,7))) invalid_weekend FROM analytics.item_observation_30d")
display(calendar_feature_check.rename(columns={'invalid_month': 'Bulan invalid', 'invalid_day': 'Hari invalid', 'invalid_weekend': 'Flag weekend tidak sesuai'}))
feature_decisions = pd.DataFrame([['Target sangat imbalance','Gunakan stratified temporal evaluation; nilai precision, recall, PR-AUC, ROC-AUC, calibration; class weight/resampling hanya pada train'],['Model dan tipe PART','Gunakan; kategori utama item'],['Client canonical','Gunakan dengan minimum support; kelompokkan kategori langka bila perlu'],['Lokasi canonical','Coverage aman; uji model dengan dan tanpa lokasi, UNKNOWN, dan missing flag'],['Umur sejak installation','Gunakan bila tidak redundan setelah seleksi korelasi'],['Bulan, kuartal, hari, weekend','Gunakan sebagai kandidat; validasi kestabilan antarperiode'],['Event/corrective/preventive 30-90-180 hari','Pilih window yang tidak redundan atau gunakan regularisasi; keputusan berdasarkan temporal validation'],['Failure 365 hari sebelumnya','Gunakan sebagai kandidat'],['Waktu sejak corrective/failure terakhir yang kosong','Missing bersifat struktural; gunakan indikator belum pernah + sentinel, bukan imputasi nol tanpa flag'],['Lama di lokasi terakhir','Gunakan hanya dari event masa lalu dan lokasi valid; sertakan missing flag'],['Fitur dengan PSI >=0,25','Jangan langsung dibuang; cek perubahan proses/data, retrain policy, dan monitoring drift'],['Interaksi model-lokasi','Uji dengan regularisasi/minimum support; jangan langsung membuat ribuan kategori'],['Status/outcome setelah snapshot','Dilarang karena leakage']], columns=['Fitur/ide','Keputusan EDA'])
display(feature_decisions)


## 8. Kesimpulan teknis dan next action

Kesimpulan berikut dibuat dari seluruh chapter kesiapan data dan fitur pada
notebook ini. Angka detail dapat berubah ketika pipeline diperbarui, sehingga
executive summary terpisah dihasilkan otomatis dari view database saat report
diekspor.


In [ ]:
weekly = cadence.loc[cadence.cadence_days.eq(7)].iloc[0]
monthly = cadence.loc[cadence.cadence_days.eq(30)].iloc[0]
ongoing_count = int(incomplete.loc[incomplete.followup_review_group.eq('LIKELY_ONGOING_0_30D'), 'failure_count'].iloc[0])
history_gap_count = int(incomplete.loc[incomplete.followup_review_group.eq('LIKELY_HISTORY_GAP_GT_180D'), 'failure_count'].iloc[0])
display(Markdown(f"""**Kesimpulan teknis**

- Snapshot 7 hari dan 30 hari sama-sama melewatkan **{int(weekly.uncaptured_failure_cycles)}** dan **{int(monthly.uncaptured_failure_cycles)}** failure.
- Snapshot mingguan memberi rata-rata **{weekly.average_positive_snapshots_per_failure:.2f}** peringatan untuk satu failure; snapshot 30 hari memberi **{monthly.average_positive_snapshots_per_failure:.2f}**.
- Ada **{ongoing_count}** failure yang kemungkinan masih berjalan karena terjadi maksimal 30 hari sebelum data berakhir.
- Ada **{history_gap_count}** failure lama yang lebih mungkin memiliki histori lanjutan tidak lengkap.
- Ada **{len(missing_onset)}** kandidat status rusak tanpa tanggal awal failure yang dapat dipercaya; kasus ini belum menjadi label utama.
- Ada **{unknown_reinstall_count} cycle reinstall** yang tidak lagi otomatis menjadi negatif dan **{coverage_unconfirmed_count} cycle** dengan coverage aktivitas belum terkonfirmasi.
- Target sangat tidak seimbang: positive rate hanya **{float(positive_row.class_percentage):.4f}%**, atau sekitar **1 positif berbanding {float(positive_row.negative_to_positive_ratio):.2f} negatif**.
- Audit korelasi menemukan **{len(redundant_pairs)} pasangan fitur** dengan |Spearman| minimal 0,80; pasangan tersebut perlu seleksi atau regularisasi, bukan dimasukkan seluruhnya tanpa evaluasi.
- Screening IV teratas adalah **{iv_result.iloc[0].feature}** dengan IV **{iv_result.iloc[0].information_value:.4f}**; hasil ini masih univariat dan wajib diuji dengan split waktu.
- Analisis PSI menemukan **{int(psi_result.psi.ge(.25).sum())} kombinasi fitur-tahun** dengan drift besar terhadap 2024; 2026 masih merupakan periode parsial.
- Unmatched lokasi hanya mengenai **{int(location_coverage.unmatched_snapshot)} snapshot ({float(location_coverage.unmatched_percentage):.4f}%)** dari data training; fitur lokasi boleh diuji dengan UNKNOWN/missing flag dan pembanding tanpa lokasi.
- Audit journal menemukan **{int(journal_quality.loc[journal_quality.check_name.eq('EXACT_LOG_DUPLICATE_EXTRA_ROWS'), 'affected_count'].iloc[0])}** baris tambahan dengan isi identik, **{int(journal_quality.loc[journal_quality.check_name.eq('INVALID_OR_FUTURE_DATE'), 'affected_count'].iloc[0])}** tanggal invalid/masa depan, dan **{int(journal_quality.loc[journal_quality.check_name.eq('LOCATION_NOT_IN_MASTER'), 'affected_count'].iloc[0])}** event lokasi non-master.
- Typo client `KERETE COMMUTER INDONESIA (KCI)` dipetakan secara fuzzy ke `KERETA COMMUTER INDONESIA (KCI)` dengan skor dan margin yang lolos batas aman.
- `GUDANG NUTECH` dipetakan ke `GUDANG NI` melalui alias kontekstual; `NOC JUANDA` tetap review karena kandidatnya ambigu.
- Lonjakan **{int(largest_spike.event_count):,} event** terbesar adalah penerimaan gudang massal, bukan installation, dismantle, atau failure.
- Lokasi hanya digunakan apabila cocok dengan master lokasi.
- Fitur kalender bulan, kuartal, hari dalam minggu, dan weekend sudah ditambahkan tanpa memakai informasi setelah snapshot.
- Untuk baseline training, snapshot 30 hari lebih ringkas: semua failure tetap tertangkap dan satu failure tidak diulang hampir empat kali.
- Saat dipakai nanti, model tetap dapat dijalankan setiap hari atau setiap ada event baru; jadwal scoring tidak harus mengikuti jarak snapshot training.
- Data siap dilanjutkan ke baseline model, sambil mereview sampel kasus histori lama dan membandingkan performa model dengan serta tanpa fitur lokasi.

**Insight bisnis (tren, risiko per model/lokasi/klien, efektivitas perbaikan, repeat failure, relokasi) ada pada notebook terpisah `01a_business_eda.ipynb`.**""".replace(',', '.')))